In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import json
from functools import partial
import torch
# torch.autograd.set_detect_anomaly(True)
from PIL import Image, ImageFilter
import cv2
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from transformers import ProcessorMixin, MllamaProcessor, AutoTokenizer, AutoImageProcessor
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from datasets import load_dataset, Dataset

from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.segmentation import SemanticSegmentationTool

from torchmetrics import JaccardIndex, PeakSignalNoiseRatio

from src.kitti_tracking import KittiDataset
from src.kitti_tracking_hf import KittiHFIterableDataset
from arc_trainer import ArcTrainer
from arc_utils import ArcCollator, ArcProcessor, ExtendedLMHead, ExtendEmbedding

/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [3]:
root_dir = "/mnt/ssd/kitti_tracking"
n_steps, n_pred_steps = 8, 0

dataset_builder = KittiHFIterableDataset(
	root_dir=root_dir,
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)
dataset = dataset_builder.to_hf_dataset()
for i, sample in enumerate(dataset):
    break

In [4]:
processor = ArcProcessor.from_data(
    model_id="meta-llama/Llama-3.2-11B-Vision-Instruct",
    path='./checkpoints',
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
collator = ArcCollator(
    processor, processor.tokenizer,
    path="./checkpoints"
)
prompts = ['people', 'vehicles']
application = "autonomous driving"

In [6]:
sem_mask.size, img.size

NameError: name 'sem_mask' is not defined

In [7]:
patch_size = 8

_imgs = sample["rgb"]

collator.inpainting.reset()
data = []
for img in _imgs:
    # 1.a/ semantic mask as ground-truth
    gt_mask = collator._get_sem_mask(img, prompts, drop_p=0)
    # gt_masks.append(gt_mask)

    # 1.b/ rand sem_mask
    sem_mask = collator._get_sem_mask(img, prompts)
    # sem_masks.append(sem_mask)
    
    # 2/ random mask as fake generated 
    rand_mask = collator._get_random_mask(img, patch_size=patch_size)
    # rand_masks.append(rand_mask)

    # 3/ prepare input
    mask = Image.composite(sem_mask, rand_mask, sem_mask)
    mask_img = Image.composite(img, Image.new("RGB", img.size, 0), mask.resize(img.size))
    recon_img = collator.inpainting(mask_img, mask)

    gray = cv2.cvtColor(np.array(recon_img), cv2.COLOR_BGR2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    print(gray.shape)
    
    h, w = gray.shape
    ph, pw = patch_size, patch_size
    
    # Trim image so it divides evenly
    h_trim = (h // ph) * ph
    w_trim = (w // pw) * pw
    
    # Reshape into patches
    lap_patches = laplacian.reshape(h_trim//ph, ph, w_trim//pw, pw)
    lap_patches = lap_patches.transpose(0, 2, 1, 3)  # (num_h, num_w, ph, pw)
    
    # Compute variance per patch
    patch_var = lap_patches.reshape(-1, ph*pw).var(axis=1)
    patch_var = patch_var.reshape(h_trim//ph, w_trim//pw)
    patch_var = np.log(patch_var + 1e-8)
    patch_var = (patch_var - patch_var.min()) / (patch_var.max() - patch_var.min() + 1e-8)

    data.append({
        "img": img.copy(),
        "gt_mask": gt_mask,
        "sem_mask": sem_mask,
        "mask": Image.composite(sem_mask, rand_mask, sem_mask),
        "rand_mask": rand_mask,
        "inp_img": recon_img,
        "lap": patch_var,
    })

(120, 320)
(120, 320)
(120, 320)
(120, 320)
(120, 320)
(120, 320)
(120, 320)
(120, 320)


In [8]:
jaccard = JaccardIndex(task="binary")

for a, b in zip(data[1:], data[:-1]):
    sem_score = jaccard(torch.tensor(np.array(b['mask']) // 255), torch.tensor(np.array(b['gt_mask'])//255))
    
    _mask = np.array(b['mask'])
    h, w = _mask.shape
    rows, cols = h // 8, w // 8
    mask_patches = _mask[:rows * patch_size, :cols * patch_size].reshape(rows, patch_size, cols, patch_size)
    mask_patches = mask_patches.mean(axis=(1, 3))
    cur_score = (b['lap'] - a['lap']) * (mask_patches / 255)

    break

sem_score, (cur_score / (mask_patches / 255).sum()).sum()

(tensor(0.1933), 0.07749926652499123)

In [9]:
collator.convert_mask_to_text(mask)

'<|begin_of_mask|><|vq_5661|><|vq_592|><|vq_4341|><|vq_3260|><|vq_6258|><|vq_2325|><|vq_4867|><|vq_6258|><|vq_2736|><|vq_5776|><|vq_5129|><|vq_5643|><|vq_2066|><|vq_559|><|vq_5002|><|vq_7223|><|vq_7986|><|vq_2513|><|vq_3260|><|vq_6258|><|vq_6258|><|vq_6258|><|vq_4952|><|vq_2241|><|vq_5579|><|vq_5661|><|vq_5002|><|vq_2241|><|vq_6796|><|vq_7986|><|vq_1863|><|vq_4341|><|vq_3071|><|vq_6258|><|vq_6258|><|vq_4952|><|vq_2241|><|vq_6320|><|vq_1192|><|vq_2736|><|vq_2045|><|vq_3551|><|vq_1022|><|vq_4707|><|vq_3260|><|vq_4867|><|vq_4670|><|vq_3454|><|vq_283|><|vq_1421|><|vq_2842|><|vq_5502|><|vq_5190|><|vq_3551|><|vq_3942|><|vq_214|><|vq_5664|><|vq_5002|><|vq_1488|><|vq_2938|><|vq_3260|><|vq_2896|><|vq_6587|><|vq_6761|><|vq_4146|><|vq_202|><|vq_4701|><|vq_3123|><|vq_6245|><|vq_6828|><|vq_1122|><|vq_1102|><|vq_4238|><|vq_2896|><|vq_2896|><|vq_6245|><|vq_1331|><|vq_6245|><|vq_5945|><|vq_3585|><|vq_283|><|vq_4594|><|vq_8165|><|vq_1480|><|vq_2051|><|vq_2325|><|vq_6258|><|vq_5229|><|vq_3929|><|vq_6510

In [21]:
collator = ArcCollator(processor, processor.tokenizer, "./checkpoints")

output, texts = collator([sample], prompts=['people', 'vehicles'], application="autonomous driving")

In [22]:
texts[0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 30 Apr 2026\n\n[{'type': 'text', 'text': 'You are a critic that requests regions in sensing data that is likely to contain salient information not already visible in the current image. You should request sensing data regions spatially relevant for the application autonomous driving. Otherwise, you should request sensing data regions that temporally uncertaint. Note that you should focus primarily on requesting relevant sensing data regions before uncertaint regions. The inputs RGB image and binary mask of size (320, 120) with patch of size (8, 8). The generated mask tokens must be enclosed within <|begin_of_mask|> and <|end_of_mask|>.'}]<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<|image|>Generate mask tokens for this image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n<|begin_of_mask|><|vq_6258|><|vq_4952|><|vq_5002|><|vq_5661|><|vq_4712|><|vq_4113|

In [27]:
text = collator.tokenizer.decode(output["input_ids"][0])

In [28]:
text

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 30 Apr 2026\n\n[{'type': 'text', 'text': 'You are a critic that requests regions in sensing data that is likely to contain salient information not already visible in the current image. You should request sensing data regions spatially relevant for the application autonomous driving. Otherwise, you should request sensing data regions that temporally uncertaint. Note that you should focus primarily on requesting relevant sensing data regions before uncertaint regions. The inputs RGB image and binary mask of size (320, 120) with patch of size (8, 8). The generated mask tokens must be enclosed within <|begin_of_mask|> and <|end_of_mask|>.'}]<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<|image|>Generate mask tokens for this image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n<|begin_of_mask|><|vq_6258|><|vq_4952|><|vq_5002|><|vq_5661|><|vq_4712|><|vq_4113|

In [24]:
tokens = collator.tokenizer.encode("hello world")
tokens

[128000, 15339, 1917]

In [26]:
text = collator.tokenizer.decode(tokens)
text

'<|begin_of_text|>hello world'